# demo_c1b

> Phase 23 (`specs/multiagent/phase23/`, experiment C1b): bridges that carry evidence, at 64 agents. Quick (the fake client and synthetic rows), the pilot, and the main run with Gate C1b (live, Claude Haiku 5.5), under `C1B_PILOT_PLAN` and `C1B_PLAN`.

Artifacts (gitignored): `results/multiagent/colony/{c1bquick, c1bpilot, c1bmain1, c1bmain2, c1bmain}/` (`runs.csv`, `turns.jsonl`, `timing.jsonl`, `metrics.json`, `report.md`, `c1b.png`, `summary.json`; `c1bmain/reused.csv`). (`c1b/` is C1's part b.) The response cache and the spend ledger are Phase 10's (`results/multiagent/llm/`), under the cap of $250 per calendar month. Rows carry C1's columns and the relay columns (`relay_mech.runC1bCell`).

- **Quick** (normal tests): the four pilot cells for 3 turns with the fake client, then the analysis, report and `c1b.png` on `syntheticRows23`, in `c1bquick/`.
- **Pilot** (`--flags llm`, `EXPLRL_LLM_LIVE=1`, `EXPLRL_C1B_STAGE=pilot`; about $2.6-3.0): `relayEvid`, `bridgeRole`, `bridgeFix` and `mirrorWindow` at seed 1890. Needs `C1B_PILOT_PLAN` frozen (after the user approved the estimate). Then `pilotCheck23`: stop and ask on any failed check, and report the projection and wait.
- **Main** (`--cell-timeout 0 --flags llm`, `EXPLRL_LLM_LIVE=1`, `EXPLRL_C1B_STAGE=main1`, then `main2`, then `main`; about $21-24): the 33 new boards in two seed batches, then the full grid from the cache ($0) with C1's 36 reused boards (`reusedRows23`), `c1bSummaries`, the report and `c1b.png`. Needs `mainReady23() == []`.

In [ ]:
import copy
import json
import os
import re
import shutil
import subprocess
import time

from explRL.bench_grid import readCSV, writeCSV
from explRL.coop_grid import coopHash
from explRL.liar_trace import cacheOnlyClient
from explRL.llm_budget import PRICES, Budget
from explRL.llm_grid import HAIKU
from explRL.llm_runs import LLMConfig
from explRL.llm_sched import mergeTiming, runWorkload, timingSummary
from explRL.relay_grid import c1bGrid, c1bPilotGrid, c1bRange, reusedCells23
from explRL.relay_mech import RELAY_COLS, reusedRows23, runC1b, runC1bCell
from explRL.relay_plan import C1B_PILOT_PLAN, C1B_PLAN, pilotCheck23
from explRL.relay_report import c1bSummaries
from explRL.relay_write import figC1b, syntheticRows23, writeC1b
from explRL.train import demo_artifact_dir

COLONY = 'multiagent/colony'
WORKERS, LLM_WORKERS = min(8, os.cpu_count() or 1), 32
LLM = demo_artifact_dir(None, 'multiagent/llm')
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
STAGE = os.environ.get('EXPLRL_C1B_STAGE', 'pilot')       # what a live run may do: pilot, main1, main2 (seed batches) or main
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=float(C1B_PLAN['cap']), ledger=str(LLM / 'spend.jsonl'))
BATCHES23 = {'main1': C1B_PLAN['seeds']['eight'][:4], 'main2': C1B_PLAN['seeds']['eight'][4:]}   # every batch runs; not analysed alone
H_CALLS = ('H23_1', 'H23_2', 'H23_3', 'H23_4')                       # P23-10: the hand-recorded calls


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def runC1bCells(mode, cells, spec, plan, workers=None):
    "C1b cells through runWorkload (runC1bCell): runs.csv, turns.jsonl, timing.jsonl and metrics.json; live runs keep *_live copies."
    live, out = spec.get('kind') == 'live', demo_artifact_dir(mode, COLONY)
    logs = out / 'turns'
    shutil.rmtree(logs, ignore_errors=True)
    t0 = time.time()
    res = runWorkload([(cells[0]['study'], cells)], spec, workers or (LLM_WORKERS if live else WORKERS), str(logs), run=runC1bCell)
    wall, rows = time.time() - t0, res['rows']
    writeCSV(rows, out / 'runs.csv')
    with open(out / 'turns.jsonl', 'w') as f:
        for p in sorted(logs.glob('*.jsonl')) if logs.exists() else []:
            f.write(p.read_text())
    timing = timingSummary(mergeTiming(logs, out / 'timing.jsonl'), wall)
    shutil.rmtree(logs, ignore_errors=True)
    meta = dict(mode=mode, planHash=coopHash(plan), commit=gitCommit(), model=HAIKU, cap=float(spec.get('cap', 0.0)), wall=wall,
                pricesChecked=PRICES[HAIKU]['checked'], cost=float(sum(r['cost'] for r in rows)), calls=int(sum(r['calls'] for r in rows)),
                spent=Budget(spec['cap'], spec['ledger']).spent() if spec.get('ledger') else 0.0)
    (out / 'metrics.json').write_text(json.dumps(dict(meta, done=res['done'], refused=res['refused'], stopped=res['stopped'], timing=timing),
                                                 indent=2, sort_keys=True, default=str) + '\n')
    if live:
        for f in ('metrics.json', 'timing.jsonl'):
            if not (out / f.replace('.', '_live.')).exists():
                shutil.copy(out / f, out / f.replace('.', '_live.'))
    print(f"{mode}: {len(rows)} rows, {meta['calls']} calls, ${meta['cost']:.2f} for these runs, ${meta['spent']:.2f} spent this month, "
          f"{wall:.0f} s; stopped: {res['stopped']}")
    return rows, meta


def analyse(mode, rows, meta, plan=C1B_PLAN, nboot=None):
    "c1bSummaries, report.md, c1b.png and summary.json in the mode's folder."
    out = demo_artifact_dir(mode, COLONY)
    S = c1bSummaries(rows, plan, nboot=nboot)
    writeC1b(out / 'report.md', S, meta, plan)
    figC1b(out, S)
    (out / 'summary.json').write_text(json.dumps(dict(meta, summary=S), indent=1, sort_keys=True, default=str) + '\n')
    return S


def mainReady23():
    """AC-8 (FR-C3, FR-C5, P23-9, P23-10) before the main run: C1B_PILOT_PLAN and C1B_PLAN frozen, C1B_PLAN on the frozen pilot
    plan's hash, the pilot's verdict and the user's approval of its projection recorded, and the hand-recorded calls for H23-1
    to H23-4 (pilot-only and every-design-helps) in C1B_PLAN['calls']."""
    c = C1B_PLAN.get('calls') or {}
    return [m for m, bad in (
        ('C1B_PILOT_PLAN is not frozen', not C1B_PILOT_PLAN['frozen']), ('C1B_PLAN is not frozen', not C1B_PLAN['frozen']),
        ("C1B_PLAN['pilotHash'] is not the frozen pilot plan's hash", C1B_PLAN['pilotHash'] != coopHash(C1B_PILOT_PLAN)),
        ("the pilot's verdict is not recorded", not C1B_PLAN.get('verdict')),
        ("the user's approval of the pilot's projection is not recorded", not (C1B_PLAN['projection'] or {}).get('approved')),
        ('the hand-recorded calls (P23-10) are missing', not all(set(H_CALLS) <= set(c.get(k) or {}) for k in ('pilotOnly', 'everyDesignHelps')))) if bad]

In [ ]:
# test_main_ready23 (AC-8, FR-C5): the main stages refuse unless every precondition holds; each one alone blocks
assert len(mainReady23()) >= 1                                  # before T6 and T7 nothing is frozen
_keep = (copy.deepcopy(C1B_PILOT_PLAN), copy.deepcopy(C1B_PLAN))
try:
    C1B_PILOT_PLAN['frozen'] = True
    calls = {h: 'yes' for h in H_CALLS}
    C1B_PLAN.update(frozen=True, pilotHash=coopHash(C1B_PILOT_PLAN), verdict='pilotCheck23 ok', projection=dict(approved='2026-11-12'),
                    calls=dict(pilotOnly=dict(calls), everyDesignHelps=dict(calls)))
    assert mainReady23() == []
    for k, v in (('frozen', False), ('pilotHash', 'x'), ('verdict', None), ('projection', dict(approved=None)),
                 ('calls', dict(pilotOnly=dict(calls), everyDesignHelps={'H23_1': 'yes'}))):
        old = C1B_PLAN[k]; C1B_PLAN[k] = v
        assert len(mainReady23()) == 1, (k, mainReady23())
        C1B_PLAN[k] = old
    C1B_PILOT_PLAN['frozen'] = False
    assert len(mainReady23()) == 2                              # unfrozen, and the hash no longer matches
finally:
    C1B_PILOT_PLAN.clear(); C1B_PILOT_PLAN.update(_keep[0]); C1B_PLAN.clear(); C1B_PLAN.update(_keep[1])
assert coopHash(C1B_PILOT_PLAN) == coopHash(_keep[0])

## Quick: the fake client and synthetic rows

In [ ]:
t0, q = time.time(), demo_artifact_dir('c1bquick', COLONY)
# the pipeline with the fake client: the four pilot cells (64 agents, relay boards) for 3 turns, with C1b's columns
cells = [dict(c, cfg=dict(copy.deepcopy(c['cfg']), turns=3)) for c in c1bPilotGrid()]
frows, _ = runC1bCells('c1bquick', cells, dict(kind='fake'), C1B_PILOT_PLAN)
assert len(frows) == 4 and all(k in r for r in frows for k in ('honestPooled', 'contamination', 'mirrorPosts', 'bridgeRelays') + RELAY_COLS)
assert {r['level'] for r in frows} == {f'{a}|independent' for a in C1B_PILOT_PLAN['cells']['pilot']}
# the analysis on synthetic rows with known effects (every planned cell)
rows = syntheticRows23()
writeCSV(rows, q / 'synthetic.csv')
S = analyse('c1bquick', rows, dict(mode='quick', planHash=coopHash(C1B_PLAN), commit=gitCommit()), nboot=200)
assert S['gate']['verdict'] == 'passes' and S['hyp']['H23_3']['supported'] and S['hyp']['H23_4']['supported']
for f in ('runs.csv', 'turns.jsonl', 'metrics.json', 'synthetic.csv', 'report.md', 'c1b.png', 'summary.json'):
    assert (q / f).stat().st_size > 0, f
for p in q.iterdir():
    if p.suffix in ('.csv', '.jsonl', '.json', '.md'):
        assert not re.search(r'sk-ant-|x-api-key|authorization|api[_-]?key', p.read_text(), re.I), p
assert time.time() - t0 < 300
print('estimate before the pilot (low, high):', c1bRange())
print(open(q / 'report.md').read()[:2000])

## Pilot (`--flags llm`, live, about $2.6-3.0)

Seed 1890, one 64-agent board with liars of each new arm (not analysed). `pilotCheck23` (P23-9): invalid replies under 5%; mirror posts; on each bridge arm, relays per bridge turn at least 0.05 and a bridge's mean input tokens per call at most 1,560; on each evidence arm, at least one relay confirmed on arrival. Then the projection of the main run beside the $28 ceiling; the verdict is always 'ask'.

In [ ]:
#| llm
if not LIVE or STAGE != 'pilot':
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_C1B_STAGE=pilot')
else:
    assert C1B_PILOT_PLAN['frozen'], 'report the estimate, get the approval, then freeze and push C1B_PILOT_PLAN before the pilot (AC-8)'
    print('pilot estimate (low, high):', c1bRange()['pilot'])
    prows, pmeta = runC1bCells('c1bpilot', c1bPilotGrid(), LIVE_SPEC, C1B_PILOT_PLAN)
    P = pilotCheck23(prows)
    demo_artifact_dir('c1bpilot', COLONY).joinpath('report.md').write_text(f"# C1b pilot (not analysed)\n\n{json.dumps(P, indent=1, default=str)}\n")
    print(f"checks {'pass' if P['ok'] else 'FAIL: stop and ask'} {P['reasons']}; cost per call by arm {P['perCall']}; projected main run "
          f"${P['projected']:.2f} against the ${P['ceiling']:.0f} ceiling: report to the user and wait for approval ({P['verdict']})")

## Main run and Gate C1b (`--cell-timeout 0 --flags llm`, live, about $21-24; then $0)

`main1` and `main2` play the 33 new boards by seed batch (1800-1803, 1804-1807); between them the only stops are 5% invalid replies and the cap. `main` replays the full grid from the cache, replays C1's 36 boards with the relay columns (`reusedRows23`, C1's columns unchanged) and writes the frozen analysis.

In [ ]:
#| llm
def reused23():
    "C1's 36 boards replayed from the cache-only client with C1b's columns, C1's columns checked (FR-C2): reused.csv."
    c1 = demo_artifact_dir('c1', COLONY)
    rows = reusedRows23(lambda cfg: cacheOnlyClient(cfg, LIVE_SPEC['cache']), readCSV(c1 / 'runs.csv') + readCSV(c1 / 'reused.csv'),
                        reusedCells23())
    writeCSV(rows, demo_artifact_dir('c1bmain', COLONY) / 'reused.csv')
    return rows


def invalidShare(rows):
    return sum(r['invalid'] for r in rows) / (sum(r['calls'] for r in rows) or 1)


if not LIVE or STAGE not in ('main1', 'main2', 'main'):
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_C1B_STAGE=main1, main2 or main')
else:
    bad = mainReady23()
    assert not bad, f'the main run waits (AC-8): {bad}'
    if STAGE in BATCHES23:
        b1 = demo_artifact_dir('c1bmain1', COLONY) / 'runs.csv'
        assert STAGE == 'main1' or (b1.exists() and invalidShare(readCSV(b1)) < C1B_PLAN['pilot']['maxInvalid']), \
            'batch 2 waits for batch 1 with invalid replies under 5% (stop and ask)'
        brows, _ = runC1bCells(f'c1b{STAGE}', [c for c in c1bGrid() if c['seed'] in BATCHES23[STAGE]], LIVE_SPEC, C1B_PLAN)
        print(f'{STAGE} (not analysed): {len(brows)} boards, invalid {invalidShare(brows):.3f}')
    else:                                                  # the full grid: every reply cached by the batches ($0; rows keep the live cost)
        for b in BATCHES23:
            f = demo_artifact_dir(f'c1b{b}', COLONY) / 'runs.csv'
            assert f.exists() and invalidShare(readCSV(f)) < C1B_PLAN['pilot']['maxInvalid'], f'run {b} first (invalid under 5%)'
        for c in c1bGrid():                                # every reply must be cached: a miss raises CacheMiss before any paid call
            cfg = LLMConfig.fromDict(c['cfg'])
            runC1b(cfg, c['seed'], cacheOnlyClient(cfg, LIVE_SPEC['cache']))
        spent = Budget(LIVE_SPEC['cap'], LIVE_SPEC['ledger']).spent()
        nrows, meta = runC1bCells('c1bmain', c1bGrid(), LIVE_SPEC, C1B_PLAN)
        assert Budget(LIVE_SPEC['cap'], LIVE_SPEC['ledger']).spent() == spent, 'the full grid must replay from the cache at $0'
        S = analyse('c1bmain', nrows + reused23(), meta)
        print(open(demo_artifact_dir('c1bmain', COLONY) / 'report.md').read()[:6000])